# Portfolio-opdracht 2: Voorspel verhuuraantallen (CityBike)

**Deel: Opdracht 2 — Time series feature engineering** · Ahmed Bashabeeb

Dit notebook is het tweede deel van ons portfolio. Aan het eind worden de delen samengevoegd in
`Portfolio2_CityBike.ipynb`. De data (`train.csv`, `test.csv`) staat in de map `data/`. Het
inladen en opschonen van de data gebeurt met de code uit Opdracht 1 (`opdracht1_eda.ipynb`).

**Inhoud**

- Voorbereiding: imports, instellingen en data uit Opdracht 1
- 2.1 Evaluatiemethode · 2.2 Stationariteit (Dickey-Fuller) · 2.3 Trend
- 2.4 Seizoenspatronen met Fourier-analyse · 2.5 Cycli, autocorrelatie en lags
- 2.6 Overige tijdgebaseerde features · 2.7 Klasse `FeatureEngineer`
- 2.8 Invloed van de features · 2.9 Gekozen features

## Voorbereiding: imports, instellingen en data uit Opdracht 1

We gebruiken de klasse `DataLoader` (paragraaf 1.2) en de functie `add_calendar_features`
(paragraaf 1.5) uit Opdracht 1, en schonen de data op zoals in paragraaf 1.9: `hum == 0`
vervangen door interpolatie en 29–30 oktober 2012 verwijderen.

In [ ]:
# Bron: Week 10 – Regression Time Series 2 (import-cel)
# Bron: Week 11 – Time series analysis 3 (imports statsmodels)

# Importeren van de packages die nodig zijn voor de feature engineering
import os
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.fft import fft
from scipy.signal import find_peaks
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.deterministic import (CalendarFourier,
                                           DeterministicProcess)
from statsmodels.tsa.stattools import adfuller
from xgboost import XGBRegressor

# Waarschuwingen onderdrukken voor een overzichtelijk notebook
warnings.filterwarnings('ignore')

# Vaste instellingen
DATA_FOLDER = os.path.join('..', 'data')     # map met de Kaggle-bestanden
RANDOM_STATE = 42                            # reproduceerbare resultaten
HORIZON = 19 * 24                            # 456 uur, zoals de Kaggle-test
HOLDOUT_START = pd.Timestamp('2012-11-12')   # laatste 19 dagen = testset
pd.set_option('display.max_colwidth', 150)
plt.rcParams['figure.figsize'] = (12, 4)
sns.set_style('whitegrid')

In [ ]:
# Bron: Opdracht 1, paragraaf 1.2 (klasse DataLoader), 1.5
#       (add_calendar_features) en 1.9 (opschonen); opdracht1_eda.ipynb

class DataLoader:
    """Leest de bestanden van de Kaggle-competitie in en schoont ze op."""

    def __init__(self, folder='.'):
        self.folder = folder

    def load(self, file_name):
        """Lees een CSV-bestand in met 'date_hour' als datetime."""
        path = os.path.join(self.folder, file_name)
        return pd.read_csv(path, parse_dates=['date_hour'])

    @staticmethod
    def clean(data, outlier_days):
        """Schoon de data op (onderbouwing: paragraaf 1.4 en 1.9).

        - sorteren op tijd en dubbele uren verwijderen;
        - luchtvochtigheid 0 (meetfout) vervangen door interpolatie;
        - uitzonderlijke dagen (outliers) verwijderen.
        """
        data = data.sort_values('date_hour')
        data = data.drop_duplicates('date_hour').copy()
        data.loc[data['hum'] == 0, 'hum'] = np.nan
        data['hum'] = data['hum'].interpolate()
        days = data['date_hour'].dt.normalize()
        data = data[~days.isin(pd.to_datetime(outlier_days))]
        return data.reset_index(drop=True)


def add_calendar_features(data):
    """Voeg tijdselementen toe die uit 'date_hour' volgen."""
    data = data.copy()
    time = data['date_hour']
    data['hour'] = time.dt.hour
    data['weekday'] = time.dt.dayofweek           # 0 = maandag, 6 = zondag
    data['day_name'] = time.dt.day_name()
    data['week'] = time.dt.isocalendar().week.astype(int)
    data['month'] = time.dt.month
    data['year'] = time.dt.year - 2011            # 0 = 2011, 1 = 2012
    data['dayofyear'] = time.dt.dayofyear
    # werkdag = maandag t/m vrijdag en geen feestdag
    is_weekday = data['weekday'] < 5
    data['workingday'] = (is_weekday & (data['holiday'] == 0)).astype(int)
    return data


# Data inladen en opschonen zoals in Opdracht 1 (paragraaf 1.9)
OUTLIER_DAYS = ['2012-10-29', '2012-10-30']
loader = DataLoader(DATA_FOLDER)
train = loader.clean(loader.load('train.csv'), OUTLIER_DAYS)
test = loader.clean(loader.load('test.csv'), outlier_days=[])
print(f'train: {len(train)} rijen | test: {len(test)} rijen')